### **Process & Thread**

1. Program : A set of instructions stored on disk to perform a specific task.
2. Process : A program in execution with its own memory, resources and execution state.
3. Thread  : The smallest unit of execution (work) within a process;  multiple threads can exist inside a single process & share the process memory.
4. Multithreading : Executing multiple threads concurrently within a single process.
5. Asynchronous Programming: Executes multiple tasks concurrently by switching between them while waiting for operations to complete, typically within a single thread.

**Process Control Block (PCB)** : PCB (Process Control Block): Core data structure used by the OS kernel to manage, track, and execute a process.
```
PCB
│
├── Process ID (PID)
├── Process State                 (Ready, Running, Waiting, Terminated)
├── Process Scheduling Info       (FCFS, SJF, Round Robin, Priority)
│
├── Program Counter
├── CPU Registers
├── Memory Information
├── I/O Status Information
└── Process Priority
```

#### **Multiprocessing** : Best for CPU Bound Tasks - Multiple processes utilize multiple CPU cores & avoid GIL limitations.

**Reason:**
- CPU-bound tasks are computation-heavy ---> in CPython, the GIL allows only one thread to execute Python bytecode at a time, limiting multithreaded performance.
- Multiprocessing uses separate processes, each with its own Python interpreter and GIL ---> enabling true parallel execution across multiple CPU cores.

**Example:**
- Process-1 → Image Processing (CPU Core 1, like one worker at a separate desk)
- Process-2 → Video Encoding (CPU Core 2, like another worker at a separate desk)
- Process-3 → ML Training (CPU Core 3, like another worker at a separate desk)
- Process-4 → Data Transformation (CPU Core 4, like another worker at a separate desk)

In [ ]:
"""
Comments:
    1. Create two processes (p1, p2).
    2. start() starts both processes.
    3. join() makes the main process wait until both processes finish.
    4. Best suited for CPU-bound tasks such as data processing, computations, and machine learning.
    
NOTE: On Windows, process creation code must be inside if __name__ == '__main__': block
For Jupyter notebooks, we use a Queue to capture child process output.
"""
import multiprocessing
from multiprocessing import Queue

def task(queue):
    total = 0
    for i in range(1000):          # CPU-intensive computation
        total += i
    result = f"Task Completed with total: {total}"
    queue.put(result)              # Send result back to main process

if __name__ == '__main__':             # Required for Windows multiprocessing
    q = Queue()                        # Create a Queue to receive outputs
    
    p1 = multiprocessing.Process(target=task, args=(q,))  # Create Process-1
    p2 = multiprocessing.Process(target=task, args=(q,))  # Create Process-2

    p1.start()                            # Start Process-1
    p2.start()                            # Start Process-2

    p1.join()                             # Wait for Process-1 to finish
    p2.join()                             # Wait for Process-2 to finish
    
    # Print results from the queue
    while not q.empty():
        print(q.get())
    
    print("All processes completed")

#### **Multithreading** : Best for I/O Bound Tasks - Threads can work while others wait for I/O.

**Reason:**
- I/O operations (API calls, DB queries, file/network requests) spend most time waiting for external resources.
- While one thread waits, others can run; in multithreading ---> Python efficiently switches between waiting threads (even with GIL).
- Result: Better responsiveness and higher throughput.

**Example:**
- Thread-1 → Waiting for API Response (like waiting in a queue)
- Thread-2 → Reading File (like waiting in another queue)
- Thread-3 → Querying Database (like waiting in a third queue)
- While one thread waits, others can continue doing useful work.

In [1]:
"""
Comments:
    1. Create two threads (t1, t2).
    2. start() starts both threads.
    3. While one thread waits (sleep/I/O), the other thread can run.
    4. join() makes the main thread wait until both threads finish.
    5. Best suited for I/O-bound tasks such as API calls, file operations, and database queries.
"""
import threading
import time

def task1():
    print("Task1 Started")
    time.sleep(2)                    # Simulates I/O wait (API/DB/File)
    print("Task1 Completed")

def task2():
    print("Task2 Started")
    time.sleep(2)                    # Simulates I/O wait (API/DB/File)
    print("Task2 Completed")

t1 = threading.Thread(target=task1)  # Create a thread for task1
t2 = threading.Thread(target=task2)  # Create a thread for task2

t1.start()                           # Start execution of task1 in a separate thread
t2.start()                           # Start execution of task2 in a separate thread

t1.join()                            # Wait for task1 thread to complete
t2.join()                            # Wait for task2 thread to complete

print("All tasks completed")

Task1 StartedTask2 Started

Task2 CompletedTask1 Completed

All tasks completed


### **Asynchronous Programming: Best for I/O-Bound Tasks — Tasks switch while waiting for I/O.**

**Reason:**
- I/O operations (API calls, database queries, file/network requests) spend time waiting for external resources.
- `async` defines a coroutine, and `await` allows it to pause while waiting for an operation, giving other tasks an opportunity to run.
- `asyncio` uses an event loop to schedule and manage asynchronous tasks.
- Result: Better responsiveness and higher throughput when handling many concurrent I/O operations.

**Example:**
- Task-1 → Waiting for an API response.
- Task-2 → Waiting for a database response.
- Task-3 → Waiting for another API response.
- While one task waits, the event loop can run another ready task.

**Main Difference:**
- Multiprocessing → Multiple processes; best suited for CPU-bound work.
- Multithreading → Multiple threads; useful for I/O-bound work.
- Asynchronous Programming → Multiple tasks managed through cooperative switching; useful for I/O-bound work, especially many concurrent operations.

In [ ]:
"""
Comments:
    1. Define two asynchronous tasks (task1, task2).
    2. async defines coroutine functions.
    3. await pauses the current coroutine while waiting, allowing other tasks to run.
    4. asyncio.gather() runs both tasks concurrently.
    5. asyncio.run() starts the event loop and executes main().
    6. Best suited for I/O-bound tasks such as API calls and network requests.
"""

import asyncio

async def task1():
    print("Task1 Started")
    await asyncio.sleep(2)             # Simulates a non-blocking I/O wait
    print("Task1 Completed")

async def task2():
    print("Task2 Started")
    await asyncio.sleep(1)             # Simulates a non-blocking I/O wait
    print("Task2 Completed")

async def main():
    await asyncio.gather(task1(), task2())  # Run both tasks concurrently

asyncio.run(main())                    # Start the event loop and execute main()